In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

#tabelas = ["chamados_hora", "base_atendentes", "base_motivos", "canais", "chamados", "clientes", "custos", "pesquisa_satisfacao"]

#### Abaixo, fazemos query sobre a tabela chamados_hora para termos uma visão geral sobre cada coluna

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Abertura_Chamado is null

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Inicio_Atendimento is null

In [0]:
%sql
select count(*) from vcredit_medallion.bronze.chamados_hora where Hora_Inicio_Atendimento is null

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Finalizacao_Atendimento is null

#### Apenas na coluna Hora_Inicio_Atendimento foram encontrados valores nulos. Vamos verificar o motivo desses valores nulos estarem aparecendo. A primeira hipótese é de que esses chamados foram abertos, mas o cliente desistiu de dar prosseguimento ao atendimento, e por isso esse valor de início está nulo. Se esse for o caso, o horário de finalização deve ser igual ao horário de abertura. Além disso, a coluna resolvido da tabela chamados deve ser falsa.  
####  
#### A segunda hipótese é de que esses chamados não são por atendimento humano, e o horário de início, apesar de não estar declarado, deve ser igual ao horário de abertura.

In [0]:
%sql
-- essa query verifica se há casos resolvidos com horário de início nulo
select * from vcredit_medallion.bronze.chamados_hora a 
inner join vcredit_medallion.silver.tb_chamados b
on a.ID_Chamado=b.id_chamado
where a.Hora_Inicio_Atendimento is null and b.resolvido="true"


In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora a 
inner join vcredit_medallion.silver.tb_chamados b
on a.ID_Chamado=b.id_chamado
where a.Hora_Abertura_Chamado=a.Hora_Finalizacao_Atendimento

#### As queries acima nos permitem concluir que a primeira hipótese está correta. Os casos de horário de início de atendimento nulo são aqueles em que o chamado não foi resolvido, uma vez que não existe caso de horário de início nulo e chamado resolvido.  
#### Além disso, a segunda query refuta a segunda hipótese, de que o horário de abertura deveria ser igual ao de finalização para os casos de horário de início nulo.

#### As tabelas que necessitam formatação são: Hora_Abertura_Chamado, Hora_Inicio_Atendimento e Hora_Finalizacao_Atendimento

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Inicio_Atendimento like "%-%"

In [0]:
df_bronze_chamados_hora = spark.table(f"{bronze_schema}.chamados_hora")

In [0]:
def normalizar_inteiro(col):
    """
    Remove tudo que não for dígito (0-9) e converte para inteiro (bigint).
    - texto sem nenhum dígito (ex.: 'NULL', '', 'abc') vira null
    - mais de 18 dígitos vira null (evita estouro do bigint)
    Aceita nome de coluna (str) ou Column.
    """
    c = F.col(col) if isinstance(col, str) else col
    digitos = F.regexp_replace(c.cast("string"), r"[^0-9]", "")
    return (
        F.when((digitos == "") | (F.length(digitos) > 18), F.lit(None).cast("long"))
         .otherwise(digitos.cast("long"))
    )

In [0]:
# dd/mm/aaaa  <qualquer coisa que não seja dígito>  hh:mm
# o \D+ cobre " às ", " �s ", " " etc. — não depende do encoding do "à"
PADRAO_DATAHORA = r"(\d{1,2})/(\d{1,2})/(\d{4})\D+(\d{1,2}):(\d{2})"

def extrair_datahora(col, prefixo):
    """
    Extrai dia, mês, ano, hora e minuto de strings tipo '12/07/2025 às 09:23:50'.
    Retorna um dict {f'{prefixo}_dia': Column, ...} para usar com df.withColumns().
    Se a string não bater com o padrão, as 5 partes vêm null.
    """
    c = F.col(col) if isinstance(col, str) else col
    s = c.cast("string")
    partes = ["dia", "mes", "ano", "hora", "minuto"]
    return {
        f"{prefixo}_{nome}": normalizar_inteiro(F.regexp_extract(s, PADRAO_DATAHORA, i))
        for i, nome in enumerate(partes, start=1)
    }

In [0]:
# Ranges possíveis para validar os campos extraídos
dia_range = [1,31]
mes_range = [1,12]
hora_range = [0,23]
minuto_range = [0,59]

# ==================================
# Extrai dia, mês, ano, hora e minuto das colunas Hora_Abertura_Chamado,
# Hora_Inicio_Atendimento e Hora_Finalizacao_Atendimento 
# ==================================
df_silver_chamados_hora = (df_bronze_chamados_hora
                           .withColumns(extrair_datahora("Hora_Abertura_Chamado", "abertura"))
                           .withColumns(extrair_datahora("Hora_Inicio_Atendimento", "inicio"))
                           .withColumns(extrair_datahora("Hora_Finalizacao_Atendimento", "fim"))
                           .drop("Hora_Abertura_Chamado", "Hora_Inicio_Atendimento", "Hora_Finalizacao_Atendimento")
                           )

display(df_silver_chamados_hora.limit(10))

In [0]:
# ======================= grava na silver =======================
(df_silver_chamados_hora.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_chamados_hora"))
print(f"\nGravado em {silver_schema}.tb_chamados_hora")